# Lesson 4 solutions 2: JIT-compiled Higgs combinatorics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import numba as nb

import awkward as ak
import vector
from hist import Hist
vector.register_awkward()

In [ ]:
events = ak.from_parquet("../data/SMHiggsToZZTo4L.parquet")

<br><br><br>

In [ ]:
eplus = events.electron[events.electron.charge > 0]
eminus = events.electron[events.electron.charge < 0]

In [ ]:
eplus_event = eplus[448]
eminus_event = eminus[448]

In [ ]:
eplus_event.show()

In [ ]:
eminus_event.show()

<br><br><br>

## Solution to exercise 1

In [ ]:
print( "e+ index  | e- index ")
print(f"Z1_i Z2_i | Z1_j Z2_j")
print( "----------+----------")
for Z1_i in range(len(eplus_event)):
    for Z2_i in range(Z1_i + 1, len(eplus_event)):
        for Z1_j in range(len(eminus_event)):
            for Z2_j in range(Z1_j + 1, len(eminus_event)):
                print(f"{Z1_i:4d} {Z2_i:4d} | {Z1_j:4d} {Z2_j:4d}")
        print( "----------+----------")

<br><br><br>

## Solution to exercise 2

In [ ]:
print( " Zsmall  |  Zbig   ")
print( "---------+---------")
for Z1_i in range(len(eplus_event)):
    for Z2_i in range(Z1_i + 1, len(eplus_event)):
        for Z1_j in range(len(eminus_event)):
            for Z2_j in range(Z1_j + 1, len(eminus_event)):
                Z1 = eplus_event[Z1_i] + eminus_event[Z1_j]
                Z2 = eplus_event[Z2_i] + eminus_event[Z2_j]
                if Z1.mass < Z2.mass:
                    Zsmall, Zbig = Z1, Z2
                else:
                    Zsmall, Zbig = Z2, Z1
                print(f"{Zsmall.mass:.5f} | {Zbig.mass:.5f}")
        print( "---------+---------")

<br><br><br>

## Solution to exercise 3

In [ ]:
from particle import Particle
from hepunits import GeV

onshell_mass = Particle.from_name("Z0").mass / GeV

In [ ]:
@nb.njit
def best_interpretation(eplus_event, eminus_event):
    best_Zsmall, best_Zbig = None, None
    for Z1_i in range(len(eplus_event)):
        for Z2_i in range(Z1_i + 1, len(eplus_event)):
            for Z1_j in range(len(eminus_event)):
                for Z2_j in range(Z1_j + 1, len(eminus_event)):
                    Z1 = eplus_event[Z1_i] + eminus_event[Z1_j]
                    Z2 = eplus_event[Z2_i] + eminus_event[Z2_j]
                    if Z1.mass < Z2.mass:
                        Zsmall, Zbig = Z1, Z2
                    else:
                        Zsmall, Zbig = Z2, Z1
                    if best_Zbig is None or abs(Zbig.mass - onshell_mass) < abs(best_Zbig.mass - onshell_mass):
                        best_Zsmall, best_Zbig = Zsmall, Zbig
    return best_Zsmall, best_Zbig

best_interpretation(eplus_event, eminus_event)

<br><br><br>

## Solution to exercise 4

In [ ]:
@nb.njit
def best_interpretation_per_event(eplus, eminus):
    zmass_small = np.empty(len(eplus), dtype=np.float64)
    zmass_big = np.empty(len(eplus), dtype=np.float64)
    higgsmass = np.empty(len(eplus), dtype=np.float64)
    for i, (eplus_event, eminus_event) in enumerate(zip(eplus, eminus)):
        best_Zsmall, best_Zbig = best_interpretation(eplus_event, eminus_event)
        if best_Zsmall is not None and best_Zbig is not None:
            zmass_small[i] = best_Zsmall.mass
            zmass_big[i] = best_Zbig.mass
            if 12 < best_Zsmall.mass < 120 and 40 < best_Zbig.mass < 120:
                higgsmass[i] = (best_Zsmall + best_Zbig).mass
            else:
                higgsmass[i] = np.nan
        else:
            zmass_small[i] = np.nan
            zmass_big[i] = np.nan
            higgsmass[i] = np.nan
    return zmass_small, zmass_big, higgsmass

zmass_small, zmass_big, higgsmass = best_interpretation_per_event(eplus, eminus)

In [ ]:
Hist.new.Regular(60, 0, 120, name="zmass_small").Regular(60, 0, 120, name="zmass_big").Double().fill(
    zmass_small=zmass_small,
    zmass_big=zmass_big,
).plot2d_full();

In [ ]:
Hist.new.Regular(100, 0, 200, name="Higgs mass").Double().fill(
    higgsmass
).plot();